# Metadata Filtering for Bedrock Managed Knowledge Bases

Metadata filtering lets you scope retrieval results to documents matching specific attributes — like department, document type, date, or confidentiality level.

### How metadata works

```
S3 Bucket
  ├── report.pdf                         ← Source document
  └── report.pdf.metadata.json           ← Metadata sidecar file
          {
            "metadataAttributes": {
              "department": "finance",
              "year": 2024,
              "confidential": false
            }
          }
```

At query time, you add a `filter` to `managedSearchConfiguration` to restrict results:
```python
response = client.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={'text': 'revenue growth'},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 5,
            'filter': {
                'equals': {'key': 'department', 'value': 'finance'}
            }
        }
    }
)
```

### Supported filter operators

| Operator | Example | Description |
|----------|---------|-------------|
| `equals` | `{'key': 'dept', 'value': 'finance'}` | Exact match |
| `notEquals` | `{'key': 'status', 'value': 'draft'}` | Exclude match |
| `greaterThan` | `{'key': 'year', 'value': 2022}` | Numeric/date comparison |
| `lessThan` | `{'key': 'year', 'value': 2025}` | Numeric/date comparison |
| `in` | `{'key': 'dept', 'value': ['finance', 'legal']}` | Match any in list |
| `startsWith` | `{'key': 'title', 'value': '10K'}` | Prefix match |
| `stringContains` | `{'key': 'tags', 'value': 'risk'}` | Substring match |
| `andAll` | `[filter1, filter2]` | All must match |
| `orAll` | `[filter1, filter2]` | Any must match |

### Supported data types

- `STRING` — text values
- `NUMBER` — numeric values
- `BOOLEAN` — true/false
- `STRING_LIST` — list of strings

### What this notebook does

1. Creates synthetic documents with `.metadata.json` sidecar files
2. Uploads to S3 and ingests into a Managed KB
3. Demonstrates filtering with different operators
4. Shows combined filters (`andAll`, `orAll`)
5. Compares filtered vs unfiltered retrieval

### Prerequisites

- AWS credentials with Bedrock and IAM permissions
- Python 3.10+

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

In [ ]:
import boto3
import sys
import time
import os
import json
import pprint

sys.path.insert(0, "../..")

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

bucket_name = f'bedrock-bmkb-metadata-{suffix}-{account_id}'
embedding_model = None  # Managed default (no extra cost)

region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'Bucket:     {bucket_name}')
print(f'Embedding:  {embedding_model or "managed default (no extra cost)"}')

## Step 2 — Create synthetic documents with metadata

We'll create 4 text documents simulating an enterprise knowledge base with different departments, years, and confidentiality levels. Each document has a `.metadata.json` sidecar file.

In [ ]:
# Synthetic documents with metadata
documents = [
    {
        'filename': 'finance_annual_report_2024.txt',
        'content': """Octank Financial Annual Report 2024

Total revenue reached $4.2 billion, representing 12% year-over-year growth.
Net income was $890 million with operating margins of 21%.
The commercial banking division contributed 45% of total revenue.
Investment management grew 18% driven by new fund launches.
We returned $1.1 billion to shareholders through dividends and buybacks.
Total assets under management reached $156 billion.""",
        'metadata': {
            'department': 'finance',
            'document_type': 'annual_report',
            'year': 2024,
            'confidential': False
        }
    },
    {
        'filename': 'finance_risk_assessment_2024.txt',
        'content': """Octank Financial Risk Assessment Q4 2024

Market risk exposure increased 8% due to volatile interest rate environment.
Credit risk: Non-performing loans at 1.2%, below industry average of 1.8%.
Operational risk: Two significant incidents requiring board notification.
Cybersecurity: Successfully defended against 3 advanced persistent threats.
Regulatory risk: New compliance requirements effective Q1 2025.
Liquidity risk: Maintained LCR ratio of 135%, well above 100% minimum.""",
        'metadata': {
            'department': 'finance',
            'document_type': 'risk_assessment',
            'year': 2024,
            'confidential': True
        }
    },
    {
        'filename': 'hr_workforce_strategy_2024.txt',
        'content': """Octank Financial Workforce Strategy 2024

Total headcount: 12,500 employees across 15 countries.
Employee retention rate improved to 91% from 87% in 2023.
Launched AI upskilling program — 4,200 employees certified.
Diversity metrics: 42% women in leadership (target: 45% by 2026).
Remote work policy: Hybrid model with 3 days in-office.
Average compensation increase: 4.5% (aligned with inflation).""",
        'metadata': {
            'department': 'hr',
            'document_type': 'strategy',
            'year': 2024,
            'confidential': False
        }
    },
    {
        'filename': 'technology_roadmap_2023.txt',
        'content': """Octank Financial Technology Roadmap 2023

Cloud migration: 78% of workloads now on AWS (up from 52% in 2022).
AI/ML initiatives: Deployed fraud detection model with 99.2% accuracy.
Data platform: Migrated to lakehouse architecture reducing query costs 60%.
API strategy: Launched open banking APIs — 2,400 partner integrations.
Security: Implemented zero-trust architecture across all production systems.
Budget: $340 million technology spend (8% of revenue).""",
        'metadata': {
            'department': 'technology',
            'document_type': 'roadmap',
            'year': 2023,
            'confidential': True
        }
    },
]

print(f'Created {len(documents)} synthetic documents:')
for doc in documents:
    m = doc['metadata']
    print(f"  {doc['filename']}")
    print(f"    dept={m['department']} | type={m['document_type']} | year={m['year']} | confidential={m['confidential']}")

## Step 3 — Upload documents + metadata sidecar files to S3

For each document, we upload:
- `document.txt` — the content
- `document.txt.metadata.json` — the metadata sidecar (must use this exact naming convention)

In [ ]:
# Create bucket
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})

# Upload documents + metadata sidecar files
s3_prefix = 'docs/'

for doc in documents:
    # Upload document content
    doc_key = f"{s3_prefix}{doc['filename']}"
    s3_client.put_object(Bucket=bucket_name, Key=doc_key, Body=doc['content'].encode())
    
    # Upload metadata sidecar (naming convention: <filename>.metadata.json)
    metadata_key = f"{doc_key}.metadata.json"
    metadata_content = json.dumps({'metadataAttributes': doc['metadata']}, indent=2)
    s3_client.put_object(Bucket=bucket_name, Key=metadata_key, Body=metadata_content.encode())
    
    print(f'  ✓ {doc_key}')
    print(f'    + {metadata_key}')

print(f'\nUploaded {len(documents)} documents + {len(documents)} metadata files to s3://{bucket_name}/{s3_prefix}')

## Step 4 — Create KB and ingest

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=f'bmkb-metadata-{suffix}',
    data_sources=[{
        'type': 'S3',
        'bucket_name': bucket_name,
        's3_prefix': s3_prefix,
    }],
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')

# Ingest
time.sleep(15)
kb.start_ingestion_job()

## Step 5 — Filter by department (equals)

Retrieve only from finance documents.

In [ ]:
runtime_client = kb.get_runtime_client()

query = 'What are the key numbers and metrics?'

# Filter: department == 'finance' only
response = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': query},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 10,
            'filter': {
                'equals': {'key': 'department', 'value': 'finance'}
            }
        }
    }
)

print(f'Query: "{query}"')
print(f'Filter: department == "finance"')
print(f'Results: {len(response["retrievalResults"])}\n')

for i, r in enumerate(response['retrievalResults'], 1):
    score = r.get('score', 0)
    text = r['content']['text'][:150]
    metadata = r.get('metadata', {})
    print(f'{i}. score={score:.4f} | dept={metadata.get("department", "?")} | type={metadata.get("document_type", "?")}')
    print(f'   {text}...')
    print()

## Step 6 — Filter by year (greaterThan)

Only return documents from 2024 onwards.

In [ ]:
response = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': 'technology and AI initiatives'},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 10,
            'filter': {
                'greaterThan': {'key': 'year', 'value': 2023}
            }
        }
    }
)

print(f'Filter: year > 2023 (only 2024+ documents)')
print(f'Results: {len(response["retrievalResults"])}\n')

for i, r in enumerate(response['retrievalResults'], 1):
    metadata = r.get('metadata', {})
    print(f'{i}. year={metadata.get("year", "?")} | dept={metadata.get("department", "?")} | {r["content"]["text"][:100]}...')

## Step 7 — Exclude confidential documents (notEquals)

Filter out sensitive content — only return non-confidential documents.

In [ ]:
response = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': 'risk and security'},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 10,
            'filter': {
                'equals': {'key': 'confidential', 'value': 'false'}
            }
        }
    }
)

print(f'Filter: confidential == false (non-confidential only)')
print(f'Results: {len(response["retrievalResults"])}\n')

for i, r in enumerate(response['retrievalResults'], 1):
    metadata = r.get('metadata', {})
    print(f'{i}. confidential={metadata.get("confidential", "?")} | dept={metadata.get("department", "?")} | {r["content"]["text"][:100]}...')

## Step 8 — Combined filters (andAll)

Multiple conditions: finance department AND year 2024 AND not confidential.

In [ ]:
response = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': 'financial performance and revenue'},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 10,
            'filter': {
                'andAll': [
                    {'equals': {'key': 'department', 'value': 'finance'}},
                    {'equals': {'key': 'year', 'value': '2024'}},
                    {'equals': {'key': 'confidential', 'value': 'false'}},
                ]
            }
        }
    }
)

print(f'Filter: department=finance AND year=2024 AND confidential=false')
print(f'Results: {len(response["retrievalResults"])}\n')

for i, r in enumerate(response['retrievalResults'], 1):
    metadata = r.get('metadata', {})
    print(f'{i}. dept={metadata.get("department")} | year={metadata.get("year")} | confidential={metadata.get("confidential")}')
    print(f'   {r["content"]["text"][:150]}...')
    print()

## Step 9 — OR filter (orAll)

Match documents from finance OR technology departments.

In [ ]:
response = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': 'budget and spending'},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 10,
            'filter': {
                'orAll': [
                    {'equals': {'key': 'department', 'value': 'finance'}},
                    {'equals': {'key': 'department', 'value': 'technology'}},
                ]
            }
        }
    }
)

print(f'Filter: department=finance OR department=technology')
print(f'Results: {len(response["retrievalResults"])}\n')

for i, r in enumerate(response['retrievalResults'], 1):
    metadata = r.get('metadata', {})
    print(f'{i}. dept={metadata.get("department")} | {r["content"]["text"][:120]}...')

## Step 10 — Filtered vs unfiltered comparison

Show how filtering narrows down results and improves relevance for specific use cases.

In [ ]:
query = 'What are the main risks and how are they managed?'

# Unfiltered
resp_unfiltered = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': query},
    retrievalConfiguration={'managedSearchConfiguration': {'numberOfResults': 10}}
)

# Filtered: finance department only
resp_filtered = runtime_client.retrieve(
    knowledgeBaseId=kb.kb_id,
    retrievalQuery={'text': query},
    retrievalConfiguration={
        'managedSearchConfiguration': {
            'numberOfResults': 10,
            'filter': {'equals': {'key': 'department', 'value': 'finance'}}
        }
    }
)

print(f'Query: "{query}"\n')
print(f'=== Unfiltered (all departments) ===')
print(f'Results: {len(resp_unfiltered["retrievalResults"])}')
depts_unfiltered = [r.get('metadata', {}).get('department', '?') for r in resp_unfiltered['retrievalResults']]
print(f'Departments in results: {depts_unfiltered}')

print(f'\n=== Filtered (finance only) ===')
print(f'Results: {len(resp_filtered["retrievalResults"])}')
depts_filtered = [r.get('metadata', {}).get('department', '?') for r in resp_filtered['retrievalResults']]
print(f'Departments in results: {depts_filtered}')

print(f'\nFiltering ensures results come only from the target department.')

## Step 11 — Cleanup

In [ ]:
# Uncomment to delete all resources
# kb.delete_kb(delete_iam=True, delete_s3_bucket=True)

## Summary

### Metadata sidecar file format

```
s3://bucket/docs/report.pdf                    ← Document
s3://bucket/docs/report.pdf.metadata.json      ← Metadata (same name + .metadata.json)
```

```json
{
  "metadataAttributes": {
    "department": "finance",
    "year": 2024,
    "confidential": false,
    "tags": ["annual", "report"]
  }
}
```

### Filter operators reference

| Operator | Use case |
|----------|----------|
| `equals` | Exact match (department, status) |
| `notEquals` | Exclusion (skip drafts, confidential) |
| `greaterThan` / `lessThan` | Date ranges, version numbers |
| `in` / `notIn` | Match/exclude multiple values |
| `startsWith` | Prefix matching (file paths, IDs) |
| `stringContains` | Substring search |
| `andAll` | All conditions must match |
| `orAll` | Any condition can match |

### Use cases

- **Multi-tenancy** — filter by `tenant_id` to scope results per customer
- **Access control** — filter by `confidentiality_level` or `department`
- **Temporal queries** — filter by `year` or `date` for recent-only results
- **Document type** — filter by `type` (policy, report, FAQ, etc.)

### Documentation

- [Include metadata in a data source](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-metadata.html)
- [Query configurations](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-test-config.html)
- [Retrieve API reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_Retrieve.html)